# Lecture 6 — Input Models & Block Encoding (UnitaryLab companion)

Part II of the course (Quantum Numerical Linear Algebra) rests on one idea: to
compute with a matrix $A$ on a quantum computer we must first *load* it. The
dominant input model is the **block encoding** — we embed a suitably normalized
copy of $A$ in the top-left corner of a larger unitary $U$ acting on
ancilla $+$ system qubits:

$$U \;=\; \begin{pmatrix} A/\alpha & \ast \\ \ast & \ast \end{pmatrix},
\qquad (\langle 0|_a \otimes I)\,U\,(|0\rangle_a \otimes I) \;=\; A/\alpha.$$

Here $\alpha \ge \|A\|$ is the **subnormalization**; every downstream algorithm
(LCU, qubitization, QSVT, QLSP) pays for a large $\alpha$ through more
repetitions. This notebook builds an exact block encoding of a small Hermitian
$A$ by hand with `Circuit.unitary`, then verifies the defining property two ways:
against the circuit matrix and by acting on $|0\rangle_a|\psi\rangle$.

Contents:
1. Setup
2. A Hermitian $A$ and its exact dilation
3. Verify the top-left block equals $A/\alpha$
4. Operational check: $U|0\rangle_a|\psi\rangle$ recovers $(A/\alpha)|\psi\rangle$
5. The role of $\alpha$
6. Your turn

In [1]:
import numpy as np
import unitarylab_algorithms as ua
from unitarylab.core import Circuit
import contextlib, io

# Suppress the algorithms' verbose logging so cell outputs stay clean.
@contextlib.contextmanager
def quiet():
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yield

np.set_printoptions(precision=4, suppress=True)
print("ready: unitarylab_algorithms loaded")

ready: unitarylab_algorithms loaded


## 2. A Hermitian $A$ and its exact dilation

For a Hermitian matrix $\tilde A = A/\alpha$ with $\|\tilde A\|_2 \le 1$, the
Hermitian dilation

$$U \;=\; \begin{pmatrix} \tilde A & \sqrt{I-\tilde A^2} \\ \sqrt{I-\tilde A^2} & -\tilde A \end{pmatrix}$$

is unitary ($U^\dagger U = I$) and is an exact $(\alpha, 1, 0)$-block encoding of
$A$: one ancilla qubit, subnormalization $\alpha$, and zero error. We take a
$2\times2$ Hermitian $A$ with $\|A\|_2 = 0.7$, so $\alpha = 1$ suffices.

In [2]:
def hermitian_dilation(Ah):
    '''Return the 2n x 2n Hermitian unitary dilation of an n-qubit Hermitian Ah with ||Ah||<=1.'''
    S = np.eye(Ah.shape[0]) - Ah @ Ah
    w, V = np.linalg.eigh(S)
    w = np.clip(w, 0, None)                 # guard tiny negative eigenvalues
    R = V @ np.diag(np.sqrt(w)) @ V.conj().T
    return np.block([[Ah, R], [R, -Ah]])

A = np.array([[0.6, 0.3], [0.3, -0.2]], dtype=complex)   # Hermitian
alpha = 1.0
assert np.allclose(A, A.conj().T), "A must be Hermitian"
print("A =\n", np.real(A))
print("||A||_2 =", round(np.linalg.norm(A, 2), 4), " -> alpha =", alpha)

Ah = A / alpha
U = hermitian_dilation(Ah)
print("\nU unitary?", np.allclose(U.conj().T @ U, np.eye(4)))

A =
 [[ 0.6  0.3]
 [ 0.3 -0.2]]
||A||_2 = 0.7  -> alpha = 1.0

U unitary? True


## 3. Verify the top-left block equals $A/\alpha$

We load $U$ onto a 2-qubit circuit (qubit 0 = system, qubit 1 = ancilla) and read
the circuit matrix with `get_matrix()`. In UnitaryLab's ordering the ancilla is
the most-significant qubit, so the ancilla-$|0\rangle$ block is the top-left
$2\times2$ submatrix — which must equal $A/\alpha$.

In [3]:
qc = Circuit(2, name="block_encoding")
qc.unitary(U, [0, 1])                       # system=q0, ancilla=q1
M = qc.get_matrix()
top_left = M[:2, :2]
print("top-left block (ancilla=0):\n", np.round(np.real(top_left), 4))
print("A/alpha:\n", np.round(np.real(A / alpha), 4))

# Exact block encoding: zero error, so a tight tolerance is appropriate.
assert np.allclose(top_left, A / alpha, atol=1e-12)
print("\nVerified: (<0|_a (x) I) U (|0>_a (x) I) = A/alpha  (error < 1e-12).")

top-left block (ancilla=0):
 [[ 0.6  0.3]
 [ 0.3 -0.2]]
A/alpha:
 [[ 0.6  0.3]
 [ 0.3 -0.2]]

Verified: (<0|_a (x) I) U (|0>_a (x) I) = A/alpha  (error < 1e-12).


## 4. Operational check: acting on $|0\rangle_a|\psi\rangle$

The block-encoding property is most useful operationally: preparing the ancilla
in $|0\rangle$ and the system in $|\psi\rangle$, applying $U$, and projecting the
ancilla back onto $|0\rangle$ yields the (subnormalized) vector
$(A/\alpha)|\psi\rangle$. We read this off the statevector directly.

In [4]:
psi = np.array([0.8, 0.6], dtype=complex)   # normalized system state
qc = Circuit(2)
qc.initialize(psi, target=[0])              # system on q0
qc.unitary(U, [0, 1])
state = qc.execute(backend="numpy", dtype=np.complex128).state.reshape(2, 2)  # [anc, sys]

anc0_block = state[0]                        # ancilla = 0 component
expected = (A / alpha) @ psi
print("U|0>_a|psi>, ancilla=0 block :", np.round(anc0_block, 4))
print("(A/alpha)|psi>               :", np.round(expected, 4))
prob_success = np.linalg.norm(anc0_block) ** 2
print("P(ancilla measured 0) =", round(prob_success, 4))

assert np.allclose(anc0_block, expected, atol=1e-12)
print("\nVerified: the ancilla-0 branch equals (A/alpha)|psi> exactly.")

U|0>_a|psi>, ancilla=0 block : [0.66+0.j 0.12+0.j]
(A/alpha)|psi>               : [0.66+0.j 0.12+0.j]
P(ancilla measured 0) = 0.45

Verified: the ancilla-0 branch equals (A/alpha)|psi> exactly.


## 5. The role of $\alpha$

If $\|A\|_2 > 1$ we cannot embed $A$ itself in a unitary — the block must be a
*contraction*. We instead block-encode $A/\alpha$ for some $\alpha \ge \|A\|_2$.
The verification then reads $\alpha \cdot (\text{top-left block}) = A$. Downstream,
the success probability of extracting $A|\psi\rangle$ scales like
$\|A|\psi\rangle\|^2/\alpha^2$, so a wasteful $\alpha$ directly inflates the query
cost. Choosing $\alpha$ close to $\|A\|_2$ is a central design concern.

In [5]:
A2 = np.array([[1.2, 0.6], [0.6, -0.4]], dtype=complex)   # ||A2||_2 = 1.4 > 1
alpha2 = 2.0                                                  # any alpha >= ||A2||_2
Ah2 = A2 / alpha2
U2 = hermitian_dilation(Ah2)

qc = Circuit(2)
qc.unitary(U2, [0, 1])
recovered = qc.get_matrix()[:2, :2] * alpha2
print("||A2||_2 =", round(np.linalg.norm(A2, 2), 4), " alpha =", alpha2)
print("alpha * (top-left block):\n", np.round(np.real(recovered), 4))
assert np.allclose(recovered, A2, atol=1e-12)
print("Verified: alpha * top-left block = A2, so U2 is an (alpha=2)-block encoding of A2.")

||A2||_2 = 1.4  alpha = 2.0
alpha * (top-left block):
 [[ 1.2  0.6]
 [ 0.6 -0.4]]
Verified: alpha * top-left block = A2, so U2 is an (alpha=2)-block encoding of A2.


## 6. Your turn
- Block-encode the Pauli $X$ (already norm 1) and confirm the top-left block is $X$.
- Take a random $4\times4$ Hermitian $A$, rescale by $\alpha = \|A\|_2$, build the
  dilation on 3 qubits (2 system + 1 ancilla), and verify the operational action on
  a random $|\psi\rangle$.
- Increase $\alpha$ beyond $\|A\|_2$ and watch $P(\text{ancilla}=0)$ shrink — this is
  the price of a loose subnormalization.

See the Lecture 6 notes for sparse-access and quantum-data-structure input models,
which give block encodings without writing $U$ out explicitly.